<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">

## Toxicity classification on molecular graphs using Graph Neural Networks (GNN) and `pytorch-geometric`

**Goal** Familiarize `pytorch-geometric` in handling GNNs and `DataLoaders`, and classify whether a molecule is toxic or not (**molecular level binary-property**)

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">
    
**Training dataset** : goal is to predict chemical toxicity using Graph Neural Networks

* ~7,800 molecules represented by SMILES strings, each with the outputs from 12 binary classification assays. Labels are either 1 = active, 0 = inactive or NaN = not tested
* Assays include nuclear receptor signaling pathways (7 assays) and stress response pathways (5 assays)
</div>

<div style="background-color: lightblue; padding: 10px; border-radius: 5px;">
    
**Tools**
* `scikit-learn`, `torch`
* **Core cheminfo**: `RDKit` **fingerprints, descriptors** to automate molecular representation and generate the input to the classifier
* `pytorch-geometric` to featurize the molecualr graphs, define the GNN layers, the head of network is a classifier
* `torch` to train and test, using **batches** and **early_stopping**
</div>

In [ ]:
# this is to install the packages on Colab
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.0.1+cu118.html
!pip install torch-geometric

!pip install rdkit-pypi

In [ ]:
import pandas as pd
import numpy as np
import copy
import torch, os, joblib, sys
import torch.nn as nn
import torch.nn.functional as F    # activation functions

import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from rdkit import Chem, DataStructs
from rdkit.Chem import Draw, Descriptors, AllChem

from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool, GraphConv

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# LB defined quantities, imported from src
from src/utils import atom_features, bond_features, graph_featurizer_pygeom
from src/utils import split_data
from src/utils import model_training_classifier, model_testing

In [ ]:
input_file = '../Classify_toxicity/data/tox21.csv'
task = 'NR-AR'
print(f'Classification task to choose = {task}')

# training parameters
n_batches = 25
n_epochs = 200
patience = 10

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
#### Read in **Tox21** dataset, pick one classification task and featurize (calculate molecular graph embeddings as `geometric` Data objects)
</div>

In [ ]:
# Tox21 da MoleculeNet, downloaded from the internet, sicne the original link/url comes with restrictions
df = pd.read_csv(input_file)

# Print columns
print(list(df.columns))  # print first few databse columns (SMILES + target)
print('Number of molecules in dataset = ' + str(df.shape[0]))
print('Number of assays available for classification tasks = ' + str(df.shape[1]-2))

if task not in list(df.columns):
    print('acthung! there is no such classification task in the input file')

In [ ]:
full_data = []

for k, smile in enumerate(df['smiles'].values):
  # check if that task label is present or not (some assays were inconclusive on some molecules)
  if np.isnan(df[task].values[k]) == False:
    full_data.append(graph_featurizer_pygeom(Chem.MolFromSmiles(smile), k, df[task].values[k], edge_attrib=None))

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
### Define train, test and validation datasets, all organized in batches

</div>

In [ ]:
print('Organize data into train, test and validation datasets, use batches...')

# split into train, test, validation set using pytorch functionalities
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

train_dataset, val_dataset, test_dataset = split_data(full_data, train_ratio, val_ratio)

# convert to torch-geometric Datasets, using batches
train_loader = DataLoader(train_dataset, batch_size=n_batches, shuffle=True) # batch size to be used, shuffle ON for training
test_loader  = DataLoader( test_dataset, batch_size=n_batches, shuffle=False)
val_loader   = DataLoader(  val_dataset, batch_size=n_batches, shuffle=False)

In [ ]:
n_samples = len(full_data)
print(f'Total nubmber of samples = {n_samples}')

in_dim = train_dataset[0].x.shape[1]
print('Embedding size of nodes = ' + str(in_dim))

print(f'Number of training samples = {len(train_dataset)}')

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
### Define the GNN Classifier for a graph-based classification; use layers available in `torch-geometric`
        `GraphConv` + '`Relu` + `GraphConv` + `relu` + `global pooling` + `classifier`
    
Compare with hard-coded GNN tools
</div>

In [ ]:
class GNNclassifier(nn.Module):
    
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        
        """
        in_dim (int): size of input node embeddings
        hidden_dim (int): size of node embeddings after passing throuhg first laye
        """

        self.conv1 = GraphConv(in_dim, hidden_dim)     # this is aready a GNN layer that performs message passing with the nearest neighobrs
        self.conv2 = GraphConv(hidden_dim, hidden_dim)
        self.classifier = torch.nn.Linear(hidden_dim, 1)  # single Linear, we can always make this more complex

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)    # the linear layer is already implemented
        x = F.relu(x) # just need to apply a non linear activation function
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = global_mean_pool(x, batch)  # graph embedding, averaging over all the noves
        x = self.classifier(x)     # logits per graph, shape [batch_size, 1]
        return x

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
### Instantiate model, optimizer and Binary Cross Entropy (w Logits) loss function; then train the model using early stopping
Please note the model we chose (GNVConv), cannot handle edge message passing
</div>

In [ ]:
# instantiate model
model = GNNclassifier(in_dim = in_dim, hidden_dim=8).to(device)  # in_dim is hard_coded, we might automate this based on the features
print(model)  # print the architecture in terms of layers and input/output sizes

# define optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
loss_fn = nn.BCEWithLogitsLoss()

model = model_training_classifier(model, optimizer, loss_fn, train_loader, val_loader, n_epochs, device, patience, early_stop = 'on')

<div style="background-color: lightgreen; padding: 10px; border-radius: 5px;">
    
### Test trained GNN model on the test set molecular graphs, compute metrics
</div>

In [ ]:
# Set model in evaluation mode
acc, pred, red, f1, auc = model_testing(model, test_loader, device, classifier = 'on')